# 실행 중인 Colab의 평가·영상 셀 복구

현재 열려 있는 학습 노트북 **아래에 코드 셀을 순서대로 복사**하세요. 이 파일을 별도 런타임에서 실행하면 기존 변수와 파일이 없습니다. 학습/수집을 중지한 상태에서 사용합니다.

기존 candidate의 model.pt와 policy_config.json을 그대로 사용합니다. 학습, GitHub 복원/업로드, candidate 재생성은 실행하지 않습니다. 첫 셀은 같은 run에서 중단 후 남은 공식 eval.py 프로세스만 종료합니다.


In [ ]:
# 1 · 기존 run/candidate 연결 + 중단 후 남은 평가 프로세스 정리
import json, os, signal, subprocess, sys, time
from pathlib import Path

if 'experiment' not in globals() or 'CFG' not in globals():
    raise RuntimeError('학습한 노트북의 같은 런타임 아래에 이 셀을 추가하세요.')
RUN_DIR = Path(experiment.run_dir)
CANDIDATE = Path(globals().get('CANDIDATE', RUN_DIR/'integrated_candidate'))
UPSTREAM = Path(CFG['repo_dir'])
manifest = json.loads((CANDIDATE/'manifest.json').read_text(encoding='utf-8'))
selected = {level:CANDIDATE/'checkpoints'/level/'model.pt'
            for level in manifest['levels']}
if not selected or any(not path.is_file() for path in selected.values()):
    raise FileNotFoundError(f'{CANDIDATE}의 기존 candidate checkpoint를 확인하세요.')
MAX_STEPS = 200
SMOKE_SEED = 61000

def stop_leftover_official_eval():
    # Linux /proc: match the exact evaluator and this run's output directory.
    # Training/collection commands and other runs cannot match these conditions.
    proc = Path('/proc')
    if not proc.is_dir():
        return
    for entry in proc.iterdir():
        if not entry.name.isdigit() or int(entry.name) == os.getpid():
            continue
        try:
            command_bytes = (entry/'cmdline').read_bytes()
            args = command_bytes.decode(errors='replace').split('\0')
            evaluator = str((UPSTREAM/'eval.py').resolve())
            outputs = [a.split('=',1)[1] for a in args if a.startswith('hydra.run.dir=')]
            if evaluator not in args or not outputs:
                continue
            if not all(Path(output).resolve().is_relative_to(RUN_DIR.resolve()) for output in outputs):
                continue
            pid = int(entry.name)
            os.kill(pid, signal.SIGTERM)
            deadline = time.monotonic()+5
            while entry.exists() and time.monotonic() < deadline:
                # Zombies have already released GPU resources; the parent reaps them.
                if (entry/'stat').read_text().split(') ',1)[1].startswith('Z'):
                    break
                time.sleep(.1)
            else:
                if entry.exists() and (entry/'cmdline').read_bytes() == command_bytes:
                    os.kill(pid, signal.SIGKILL)
            print('중단 후 남은 이 run의 공식 평가 프로세스 종료:', pid)
        except (FileNotFoundError, ProcessLookupError, PermissionError):
            continue

stop_leftover_official_eval()
print('학습 파일 유지:', RUN_DIR)
print('기존 candidate 그대로 사용:', list(selected))


In [ ]:
# Colab inline 영상 player · 한 번만 실행
from IPython.display import Video, display

VIDEO_LEVEL = next(iter(selected))  # 'easy', 'medium', 'hard' 중 현재 생성된 난이도
VIDEO_WIDTH = 960
DOWNLOAD_VIDEO = False

def show_official_video(label, level=None):
    level = level or VIDEO_LEVEL
    if level not in selected:
        raise ValueError(f'{level} checkpoint가 없습니다. 가능한 난이도: {list(selected)}')
    folder = RUN_DIR/level/'integrated_official_eval'/label/'videos'
    videos = sorted(folder.rglob('*.mp4'), key=lambda path:path.stat().st_mtime)
    if not videos:
        raise FileNotFoundError(f'{folder}에 MP4가 없습니다. 바로 앞 평가 셀을 먼저 실행하세요.')
    video = videos[-1]
    print(f'[{level}/{label}] {video.name} · {video.stat().st_size/1024**2:.1f} MiB')
    # Older Colab IPython checks os.path.exists(data) before filename.
    display(Video(str(video), embed=True, width=VIDEO_WIDTH))
    if DOWNLOAD_VIDEO:
        from google.colab import files
        files.download(str(video))
    return video

def show_all_official_videos(label):
    for level in selected:
        try:
            show_official_video(label, level)
        except FileNotFoundError as error:
            print(error)

show_all_official_videos('smoke')


In [ ]:
# 공식 eval.py · 1 episode smoke
import os, signal, subprocess, sys
RESULTS = RUN_DIR
SMOKE_CONFIG = RUN_DIR/'integrated_smoke_eval.yaml'
SMOKE_CONFIG.write_text('eval:\n  n_episodes: 1\n  seeds: ['+str(SMOKE_SEED)+']\n', encoding='utf-8')
UPSTREAM = Path(CFG['repo_dir'])

def run_official(level, eval_config, label, candidate=None):
    candidate = Path(candidate) if candidate is not None else CANDIDATE
    output = RUN_DIR/level/'integrated_official_eval'/label
    output.mkdir(parents=True, exist_ok=True)
    command = [sys.executable, str(UPSTREAM/'eval.py'), 'difficulty='+level,
        'obs_mode=state', 'policy=stage_chunk_policy:load_policy',
        'checkpoint='+str(candidate/'checkpoints'/level/'model.pt'),
        'eval_config='+str(eval_config), 'max_episode_steps='+str(MAX_STEPS),
        'hydra.run.dir='+str(output)]
    child_env = dict(os.environ)
    child_env['PYTHONPATH'] = str(candidate)+os.pathsep+str(UPSTREAM)+os.pathsep+child_env.get('PYTHONPATH','')
    log = output/'official_eval.log'
    with log.open('w', encoding='utf-8') as handle:
        process = subprocess.Popen(command, cwd=UPSTREAM, env=child_env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, errors='replace', bufsize=1)
        try:
            for line in process.stdout:
                handle.write(line)
                handle.flush()
                print(line, end='', flush=True)
            code = process.wait()
        finally:
            # Colab's stop button interrupts the kernel, not its GPU subprocess.
            if process.poll() is None:
                previous_sigint = signal.signal(signal.SIGINT, signal.SIG_IGN)
                try:
                    process.terminate()
                    try:
                        process.wait(timeout=5)
                    except subprocess.TimeoutExpired:
                        process.kill()
                        process.wait()
                finally:
                    signal.signal(signal.SIGINT, previous_sigint)
            process.stdout.close()
    if code:
        raise RuntimeError(f'official eval failed ({level}); {log} 확인')
    return log


def run_missing_smoke():
    for level in selected:
        folder = RUN_DIR/level/'integrated_official_eval/smoke/videos'
        if any(path.stat().st_size > 0 for path in folder.rglob('*.mp4')):
            print(level, '기존 smoke 영상 유지')
            continue
        run_official(level, SMOKE_CONFIG, 'smoke')
    show_all_official_videos('smoke')

# 영상이 없는 난이도만 공식 평가합니다. GitHub 업로드를 호출하지 않습니다.
run_missing_smoke()


In [ ]:
import shutil
# 현재 run 전체 PC 백업 · 중단 복구용 optimizer/RNG/recovery 데이터 포함
# 학습/수집이 끝나거나 중단되어 파일 기록이 멈춘 상태에서 실행합니다.
from google.colab import files
checkpoints = list(RUN_DIR.glob('*/checkpoints/latest.pt'))
if not checkpoints:
    raise FileNotFoundError(f'{RUN_DIR}에 latest.pt가 없습니다.')
backup = shutil.make_archive(str(RUN_DIR.parent/(RUN_DIR.name+'_backup')), 'zip',
                             root_dir=RUN_DIR.parent, base_dir=RUN_DIR.name)
print('복구용 checkpoint:', [str(path) for path in checkpoints])
print('전체 run 백업 ZIP:', backup)
files.download(backup)
